# `db.ipynb` - Conexión a PostgreSQL y auditoría

`conectar_postgres()` abre una conexión nueva con las credenciales de `config.ipynb`. `registrar_inicio_ejecucion` / `registrar_fin_ejecucion` escriben y actualizan la fila de auditoría en `tb_ejecuciones_etl` (timestamps, contadores de leídos/insertados/cuarentena, origen de la demografía y estado final `OK` / `ERROR`).

> Depende de `config.ipynb`. Requiere que el contenedor de PostgreSQL (`docker compose up -d`) esté levantado para que la celda de prueba funcione.

In [ ]:
import logging
from datetime import datetime

import psycopg2

## Conexión

In [ ]:
def conectar_postgres():
    """Abre una conexion nueva a PostgreSQL con las credenciales de config.ipynb."""
    return psycopg2.connect(
        host=DB_HOST, port=DB_PORT, dbname=DB_NAME,
        user=DB_USER, password=DB_PASSWORD,
    )

## Auditoría de ejecuciones

In [ ]:
def registrar_inicio_ejecucion(fecha_inicio: datetime) -> int:
    """Inserta la fila de auditoria de una nueva ejecucion y devuelve su id."""
    conn = conectar_postgres()
    cur = conn.cursor()
    cur.execute(
        """
        INSERT INTO tb_ejecuciones_etl (fecha_inicio, estado)
        VALUES (%s, 'EN_PROCESO') RETURNING id_ejecucion;
        """,
        (fecha_inicio,),
    )
    id_ejecucion = cur.fetchone()[0]
    conn.commit()
    cur.close()
    conn.close()
    return id_ejecucion


def registrar_fin_ejecucion(id_ejecucion, fecha_inicio, fecha_fin, leidos, insertados,
                             en_cuarentena, origen_poblacion, estado, observaciones=None):
    """Actualiza la fila de auditoria con el resultado final de la ejecucion."""
    conn = conectar_postgres()
    cur = conn.cursor()
    cur.execute(
        """
        UPDATE tb_ejecuciones_etl
           SET fecha_fin = %s, registros_leidos = %s, registros_insertados = %s,
               registros_cuarentena = %s, origen_datos_demografia = %s,
               estado = %s, observaciones = %s
         WHERE id_ejecucion = %s;
        """,
        (fecha_fin, leidos, insertados, en_cuarentena, origen_poblacion, estado, observaciones, id_ejecucion),
    )
    conn.commit()
    cur.close()
    conn.close()

## Prueba rápida

Comprueba solo la conexión (`SELECT version();`), sin tocar ninguna tabla - para confirmar que las credenciales y el puerto son correctos antes de correr el pipeline completo. Si el contenedor de Docker todavía no está levantado, avisa con un mensaje claro en vez de romper la ejecución del notebook.

In [ ]:
try:
    conn_prueba = conectar_postgres()
    cur_prueba = conn_prueba.cursor()
    cur_prueba.execute("SELECT version();")
    print("Conexion OK ->", cur_prueba.fetchone()[0])
    cur_prueba.close()
    conn_prueba.close()
except psycopg2.OperationalError as error:
    print("No se pudo conectar a PostgreSQL. Comprueba que 'docker compose up -d' este levantado.")
    print(f"Detalle: {error}")